🔥 전체 흐름 한 번에 정리
Encoder:
  문장 → Embedding → LSTM → (state_h, state_c)

Decoder:
  (state_h, state_c) + 입력 → LSTM → Dense → 단어 예측


💡 핵심 이해 포인트

✔ Encoder = 문장 “요약기”
✔ Decoder = “문장 생성기”
✔ state = 의미 전달 핵심
✔ softmax = 다음 단어 확률

[ 입력 문장 ]
     |
     v
. . . . . . . . . . . . . . .
     Encoder
. . . . . . . . . . . . . . .
     |
     |  (state_h, state_c)
     v
=============================
   context vector (요약)
=============================
     |
     v
. . . . . . . . . . . . . . .
     Decoder
. . . . . . . . . . . . . . .
     |
     v
[ 출력 문장 ]

입력:   I   am   student
        |    |      |
        v    v      v
      Embedding Layer
              |
              v
        LSTM Encoder
              |
              v
     state_h, state_c
              |
              | (점선 연결 = 의미 전달)
              v
        Decoder 초기 상태
              |
              v
    <START> → 나는 → 학생 → 이다
              |
              v
        출력 문장 생성

In [ ]:
#!pip install tensorflow
# 👉 설명
# Model → 전체 신경망을 묶는 클래스 (Functional API)
# Input → 모델 입력 정의
# LSTM → 시퀀스를 처리하는 RNN 구조
# Embedding → 단어를 벡터로 변환
# Dense → 출력층 (다음 단어 예측)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense

In [14]:
# latent_dim → LSTM이 기억하는 “정보 크기 (hidden state 크기)”
# vocab_size → 전체 단어 개수 (예: 10,000개 단어 사전)
latent_dim = 256   # LSTM이 기억하는 벡터 크기 (hidden state 크기)
vocab_size = 10000  # 전체 단어 사전 크기

In [ ]:
#Encoder 입력
#Encoder의 입력 문장 정의
#None → 문장 길이는 가변 (예: 5단어, 10단어 다 가능)
encoder_inputs = Input(shape=(None,))

# 👉 설명
# 단어를 숫자 → 벡터로 변환
# 예:
# "I" → [0.2, 0.8, ...]
enc_emb = Embedding(input_dim=vocab_size, output_dim=latent_dim)(encoder_inputs)

# LSTM 생성
# return_state=True → 마지막 hidden state를 반환 (중요)
encoder_lstm = LSTM(latent_dim, return_state=True)

# encoder_outputs → 전체 시퀀스 출력 (우리는 안 씀)
# state_h → hidden state (기억 벡터)
# state_c → cell state (장기 기억)
# Encoder는 문장을 압축해서 state_h, state_c만 전달
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)

#Decoder로 전달할 “문장 요약 정보”
encoder_states = [state_h, state_c]

In [ ]:
# Decoder
# Decoder에 들어갈 입력 문장
# 학습 시: 정답 문장을 넣음 (Teacher forcing)
decoder_inputs = Input(shape=(None,))

dec_emb_layer = Embedding(input_dim=vocab_size, output_dim=latent_dim)

# Decoder도 단어 → 벡터 변환
# Encoder와 동일한 방식
dec_emb = dec_emb_layer(decoder_inputs)

#Decoder LSTM
# return_sequences=True
# → 모든 시간 step 출력 (문장 생성해야 하니까)
# return_state=True
# → 상태도 같이 반환
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
# 👉 설명
# 핵심 부분 🔥
# Decoder 초기 상태를 Encoder 결과로 설정
# Encoder가 만든 “문장 의미”를 기반으로 문장 생성 시작
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

# 👉 설명
# 다음 단어 예측
# softmax → 확률 출력
decoder_dense = Dense(vocab_size, activation="softmax")

#LSTM 출력 → 단어 확률로 변환
decoder_outputs = decoder_dense(decoder_outputs)

In [ ]:
# 입력 2개:
# Encoder 입력
# Decoder 입력
# 출력:
# 다음 단어 확률
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
# optimizer:
# Adam (기본 최고 성능)
# loss:
# sparse_categorical_crossentropy
# → 정수 라벨 기반 단어 예측
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 256) │  2,560,000 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, None, 256) │  2,560,000 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 256),     │    525,312 │ embedding_2[0][0] │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, None,     │    525,312 │ embedding_3[0][0… │
│                     │ 256), (None,      │            │ lstm_2[0][1],     │
│                     │ 256), (None,      │            │ lstm_2[0][2]      │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  2,570,000 │ lstm_3[0][0]      │
│                     │ 10000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,740,624 (33.34 MB)

 Trainable params: 8,740,624 (33.34 MB)

 Non-trainable params: 0 (0.00 B)